## Cálculo dos indicadores

Nesse notebook são calculadas as variáveis de percentual de votos dos candidatos e de famílias beneficiárias do Programa Bolsa Família (PBF)

In [1]:
from utils.load_csv import load_csv
from utils.save_csv import save_csv

In [2]:
df_resultados_2026_filtrado = load_csv('resultados_2026_filtrado.csv')

df_observasampa_filtrado = load_csv('observasampa_filtrado.csv')

1 - Percentual de votos válidos dos candidatos à presidência no segundo turno por local de votação do município de São Paulo

In [3]:
VOTO_BRANCO, VOTO_NULO = 95, 96

chaves = ["NR_ZONA", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO", "DS_LOCAL_VOTACAO_ENDERECO"]

candidatos = df_resultados_2026_filtrado[~df_resultados_2026_filtrado["NR_VOTAVEL"].isin([VOTO_BRANCO, VOTO_NULO])]

nomes = list(candidatos["NM_VOTAVEL"].unique())

df_votos_local = (candidatos
                  .pivot_table(index=chaves, columns="NM_VOTAVEL", values="QT_VOTOS",
                               aggfunc="sum", fill_value=0)
                               .reset_index())

df_votos_local.columns.name = None

df_votos_local["total_vt_val"] = df_votos_local[nomes].sum(axis=1)

total = df_votos_local["total_vt_val"].where(df_votos_local["total_vt_val"] > 0)

for nome in nomes:
    df_votos_local[f"PCT_{nome}"] = (df_votos_local[nome] / total * 100).round(2)

df_votos_local = df_votos_local.sort_values(["NR_ZONA", "NR_LOCAL_VOTACAO"]).reset_index(drop=True)

df_votos_local = (df_votos_local
                  .rename(columns={
                      "NR_ZONA": "nr_zona",
                      "NR_LOCAL_VOTACAO": "nr_local",
                      "NM_LOCAL_VOTACAO": "nm_local",
                      "DS_LOCAL_VOTACAO_ENDERECO": "end_local",
                      "FLAVIO NANTES BOLSONARO": "flavio",
                      "LUIZ INÁCIO LULA DA SILVA": "lula",
                      "PCT_FLAVIO NANTES BOLSONARO": "pct_flavio",
                      "PCT_LUIZ INÁCIO LULA DA SILVA": "pct_lula",}))

2 - Indicador de famílias beneficiárias do PBF por mil habitantes, por distrito da capital paulista

In [4]:
df_observasampa_filtrado['pbf_mil_hab'] = (df_observasampa_filtrado['familias_pbf'] / df_observasampa_filtrado['pop_total']) * 1000

In [5]:
save_csv(df_votos_local, 'votos_local.csv')

save_csv(df_observasampa_filtrado, 'df_observasampa_filtrado')

Base salva em data\votos_local.csv
Base salva em data\df_observasampa_filtrado
